# Clisense — ML-Powered Predictive Climate Intelligence
### ALU Mission Capstone 2026 — Machine Learning Track
**Student:** H. Ayomide Agbaje | **Supervisor:** Ndinelao Iitumba

This notebook covers data generation, exploratory data analysis, feature
engineering, model training, and evaluation for the Clisense climate threat
classifier. All code here calls the exact same functions used by the deployed
Streamlit app and FastAPI backend (`app/model_core.py`), so results here are
reproducible and consistent with the live system — see `BUGFIX_REPORT.md` for
why that single-source-of-truth design matters.

**Honesty disclosure:** the dataset is synthetic, generated from climatological
normals rather than pulled live from a sensor/satellite feed. See the main
README's "Limitations" section.

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../app"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

import model_core as mc

print("Loaded model_core from app/. States:", mc.STATES)
print("Feature count:", mc.N_FEATURES)
print("Classes:", mc.LABEL_CLASSES)

## 2. Data Generation

Generates the full 18,530-row synthetic dataset. Dry-season baseline rainfall
per state is deliberately close to zero (see `model_core.DRY_BASE`) to produce
a genuine drought signal — an earlier version of this generator used dry-season
baselines of 5-10mm/day, which never actually crossed the drought threshold
even accumulated over 30 days. See `BUGFIX_REPORT.md`.

In [ ]:
df = mc.generate_synthetic_dataset(n=18530, seed=42)
print(f"{len(df):,} rows generated")
df.head()

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df, x="rainfall_mm", hue="season", bins=40, ax=axes[0])
axes[0].set_title("Daily Rainfall Distribution by Season")
sns.boxplot(df, x="state", y="rainfall_mm", ax=axes[1])
axes[1].set_title("Rainfall Distribution by State")
plt.tight_layout()
plt.show()

In [ ]:
num_cols = ["rainfall_mm","temp_c","humidity_pct","rain_7d","rain_30d","rain_anomaly","dry_spell_days"]
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
pivot = df.groupby(["month", "threat_label"]).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(pivot.T, cmap="YlOrRd", annot=True, fmt="d", ax=ax)
ax.set_title("Seasonal Threat Distribution by Month")
plt.tight_layout()
plt.show()

counts = df["threat_label"].value_counts()
print(counts)
print((counts / counts.sum() * 100).round(1))

Observed class balance (seed 42, n=18,530): **Normal 42.6%**, **Flood Risk
41.3%**, **Drought Risk 16.1%**. Drought Risk concentrates in the Nov-Mar dry
season across all five states; Flood Risk concentrates in the Apr-Oct wet
season, especially in Benue (Southern Guinea Savanna). This matches NIMET's
published seasonal calendar for these agro-ecological zones.

## 4. Feature Engineering

16 engineered features (see `model_core.FEATURE_NAMES`): raw rainfall,
temperature, humidity, wind speed; 7-day and 30-day rolling rainfall totals;
a rainfall anomaly relative to the 30-day mean; a dry-spell-days proxy; a
temperature anomaly; cyclical sine/cosine encodings of month and day-of-year
(so December and January are treated as adjacent, which raw integer months
are not); and encoded state, season, and agro-ecological zone.

In [ ]:
le_state = mc.LabelEncoder().fit(mc.STATES)
le_season = mc.LabelEncoder().fit(["dry", "wet"])
le_zone = mc.LabelEncoder().fit(sorted(set(mc.ZONE_MAP.values())))
X = mc.engineer_features(df, le_state, le_season, le_zone)
y = df["threat_label"].map(mc.LABEL_TO_IDX).values
print("Feature matrix shape:", X.shape)
print("Feature names:", mc.FEATURE_NAMES)

## 5. Model Architecture and Training

**Algorithm:** XGBoost multi-class classifier (`objective="multi:softprob"`).

**Hyperparameters:** 400 trees, max depth 6, learning rate 0.05, subsample 0.9,
colsample_bytree 0.9, L1 penalty (reg_alpha) 0.1, L2 penalty (reg_lambda) 1.0.

**Why XGBoost:** trains in seconds on this dataset size, handles the class
imbalance well with sample weighting, gives interpretable feature importances,
and does not require GPU infrastructure — appropriate for an MVP that a
supervisor or NGO partner needs to be able to reproduce on a laptop.

**Train/test split:** 80/20 stratified split, plus 5-fold stratified
cross-validation to confirm the result is not an artefact of a favourable split.

In [ ]:
bundle, metrics, _ = mc.train(df, seed=42)
mc.save_bundle(bundle, metrics)

print("Algorithm:", metrics["algorithm"])
print(f"Test accuracy:      {metrics['accuracy']*100:.2f}%")
print(f"Weighted F1:         {metrics['weighted_f1']:.4f}")
print(f"Weighted recall:     {metrics['weighted_recall']:.4f}")
print(f"Weighted precision:  {metrics['weighted_precision']:.4f}")
print(f"5-fold CV F1 mean:   {metrics['cv_f1_mean']:.4f} (+/- {metrics['cv_f1_std']:.4f})")

## 6. Evaluation

In [ ]:
cm = np.array(metrics["confusion_matrix"])
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=mc.LABEL_CLASSES, yticklabels=mc.LABEL_CLASSES, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix (Accuracy {metrics['accuracy']*100:.2f}%)")
plt.tight_layout()
plt.show()

In [ ]:
imp = metrics["feature_importances"]
names, values = list(imp.keys()), list(imp.values())
idx = np.argsort(values)[-10:]
fig, ax = plt.subplots(figsize=(7, 6))
ax.barh([names[i] for i in idx], [values[i] for i in idx], color="steelblue")
ax.set_title("Top 10 Feature Importance (XGBoost)")
plt.tight_layout()
plt.show()

**Result:** `rain_30d`, `humidity_pct`, and `dry_spell_days` dominate feature
importance, consistent with the agronomic reasoning that drought and flood
signals build over days-to-weeks rather than appearing in a single day's
reading. Full per-class precision/recall/F1 is available in
`metrics["classification_report"]` and in `models/model_metadata.json` after
running `mc.save_bundle`.

## 7. Deployment

This exact `train()` / `save_bundle()` / `load_or_train()` pipeline is what
runs inside `app/streamlit_app.py` (Streamlit Cloud) and `app/api.py` (Railway,
FastAPI) — there is no separate "notebook version" of the model that could
drift from what is actually deployed. See `DEPLOYMENT_GUIDE.md` for the full
deployment steps and `testing_screenshots/` for live verification evidence.